# Revision figures: regenerate every changed figure without the full pipelines

Run from the `JBI` folder (next to `revision_utils.py`). Each section is independent; run only what you need.
Every figure is written to `Revision/figures_regenerated/` under the file name the `.tex` files use.
Nothing in `Revision/` is overwritten unless you set `COPY_INTO_PLACE = True` in the last cell.

| Section | Figure | Input (no pipeline rerun) | First run |
|---|---|---|---|
| A1 | Fig 2a, respiratory concurrent calibration | `RF_PEMDT_output/validation_calibrated/calibrated_patients.csv` | seconds |
| A2 | Fig 3a, respiratory LLR vs SHAP | `RF_PEMDT_output/RF_dashboard_cohort.csv` (same code as the published standalone cell) | a few minutes |
| A3 | Fig S1 left, pre-onset AUROC by lead time | `RF_PEMDT_output/validation_horizon/fitted_hmm_horizon.csv` | seconds |
| B1 | Fig 3b, AKI LLR vs SHAP | `derived/aki_cohort_v2.parquet` (same code as the published standalone cell) | a few minutes |
| B2 | Fig S2 right, nephrotoxin association | cohort parquet + `diagnoses_icd`; published JAX engine (fast batch run, checked against the published per-stay call) | about 5 to 10 minutes |
| B3 | Fig 2b and Fig S5 top | outputs of the R8 cell in `AKI_revision_IBMED_v2.ipynb` | seconds |
| C1 | Fig 3c, cirrhosis LLR vs SHAP | cirrhosis Steps 1 and 2 (cohort, labs, HMM fit) and the explainability cell, executed verbatim | 10 to 30 minutes (reads `labevents`) |

A2, B1, B2 and C1 cache their small result tables after the first run, so changing a title later takes seconds
(set `RECOMPUTE = True` to rebuild). Each section prints a check against the published value.
Derived data stay in the project folders (git-ignored); only figures and aggregate tables are written.

In [10]:
# ---------------------------------------------------------------- 0 · setup
import os, sys, json, shutil, warnings, contextlib, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, norm
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore")

JBI = Path.cwd() if (Path.cwd() / "revision_utils.py").exists() else Path.cwd().parent
sys.path.insert(0, str(JBI))
import revision_utils as ru

AKI_DIR, RESP_DIR, CIRR_DIR = JBI / "PEMDT-AKI", JBI / "PEMDT - Respiratory", JBI / "PEMDT-cirrhosis"
STAGE = JBI / "Revision" / "figures_regenerated"; STAGE.mkdir(parents=True, exist_ok=True)
RECOMPUTE = True                       # True = ignore the cached result tables and recompute
BLUE, GREEN, GREY = "#1f77b4", "#2ca02c", "#9aa0a6"

try:
    import shap as _shap; _HS = True
except Exception:
    _HS = False
    print("WARNING: shap is not installed. The published figures used SHAP; the fallback (linear surrogate) "
          "will NOT reproduce Figure 3. Install shap before running A2, B1 and C1.")

EXPECT = {"Fig 3a Spearman": 0.571, "Fig 3b Spearman": 0.800, "Fig 3c Spearman": 0.842,
          "S2 naive": 0.078, "S2 IPW": 0.030, "S2 AIPW": 0.033}


def check(name, got, tol=0.0015):
    exp = EXPECT[name]
    ok = abs(got - exp) <= tol
    print(f"  check {name}: {got:+.3f} (published {exp:+.3f}) -> {'OK' if ok else 'DIFFERS, stop and compare'}")
    return ok


@contextlib.contextmanager
def in_dir(p):
    old = Path.cwd(); os.chdir(p)
    try:
        yield
    finally:
        os.chdir(old)


print("JBI:", JBI, "| output:", STAGE, "| shap:", _HS)

JBI: /Users/sumiyasayeed/Downloads/MSc/PEMDT - Diseases/JBI | output: /Users/sumiyasayeed/Downloads/MSc/PEMDT - Diseases/JBI/Revision/figures_regenerated | shap: True


In [11]:
# ------------------------------------- A1 · Figure 2a: respiratory failure, concurrent calibration
# Same calculation as the published calibration cell (RF-PEMDT.ipynb, cell 10): out-of-fold
# recalibrated patient probabilities, 10 quantile bins. Only the title changes.
from sklearn.calibration import calibration_curve
cp = pd.read_csv(RESP_DIR / "RF_PEMDT_output" / "validation_calibrated" / "calibrated_patients.csv")
fp_, mp_ = calibration_curve(cp["label"], cp["calibrated_prob"], n_bins=10, strategy="quantile")
plt.figure(figsize=(6, 6))
plt.plot(mp_, fp_, "o-"); plt.plot([0, 1], [0, 1], "k--")
plt.title("Respiratory failure (concurrent)"); plt.xlabel("Mean predicted"); plt.ylabel("Observed")
plt.tight_layout(); plt.savefig(STAGE / "figure_1.png", dpi=130); plt.close()
print("Figure 2a ->", STAGE / "figure_1.png", f"({len(cp):,} patients)")

Figure 2a -> /Users/sumiyasayeed/Downloads/MSc/PEMDT - Diseases/JBI/Revision/figures_regenerated/figure_1.png (65,363 patients)


In [12]:
# -------------------------------------------- A2 · Figure 3a: respiratory LLR vs SHAP
# Verbatim logic of "Standalone · Respiratory LLR vs SHAP" (RF-PEMDT.ipynb); only the title changes.
# Reading only the needed columns does not change the 300,000-row sample (it depends on row count and order).
CACHE_A2 = RESP_DIR / "RF_PEMDT_output" / "explainability" / "llr_vs_shap_values.csv"
LIK_FEATURES = ["SpO2", "Resp_rate", "PF_ratio", "PaCO2", "pH", "HR_bpm", "MAP", "Lactate"]
if CACHE_A2.exists() and not RECOMPUTE:
    c = pd.read_csv(CACHE_A2, index_col=0)
else:
    FEATURE_WEIGHTS = {"SpO2": 1.6, "Resp_rate": 1.4, "PF_ratio": 1.8, "PaCO2": 1.3,
                       "pH": 1.1, "HR_bpm": 0.8, "MAP": 0.9, "Lactate": 1.0}
    STATE_Z = {"SpO2": [0.0, -1.5, -3.0], "Resp_rate": [0.0, 1.5, 3.0], "PF_ratio": [0.0, -1.2, -2.6],
               "PaCO2": [0.0, 0.8, 2.2], "pH": [0.0, -0.8, -2.0], "HR_bpm": [0.0, 0.8, 1.6],
               "MAP": [0.0, -0.6, -1.4], "Lactate": [0.0, 0.8, 1.8]}
    STATE_SCALE = [1.0, 1.3, 1.6]
    _path = RESP_DIR / "RF_PEMDT_output" / "RF_dashboard_cohort.csv"
    _cols = pd.read_csv(_path, nrows=0).columns
    d = pd.read_csv(_path, low_memory=False,
                    usecols=[f for f in LIK_FEATURES if f in _cols] + ["p_stable", "p_compromise", "p_failure"])
    if len(d) > 300_000:
        d = d.sample(300_000, random_state=0).reset_index(drop=True)
    feats = [f for f in LIK_FEATURES if f in d.columns]
    X = d[feats].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    for f in feats:
        lo, hi = X[f].quantile([0.001, 0.999]); X[f] = X[f].clip(lo, hi)
    X = X.fillna(X.median())
    Yv = (d[["p_stable", "p_compromise", "p_failure"]].values.argmax(1) == 2).astype(int)
    GS = {f: (float(X[f].mean()), float(X[f].std() or 1.0)) for f in feats}
    llr = {}
    for f in feats:
        mu, sd = GS[f]; z = (X[f].to_numpy() - mu) / sd; w = FEATURE_WEIGHTS[f]
        llr[f] = float(np.mean(np.abs(w * norm.logpdf(z, STATE_Z[f][2], STATE_SCALE[2])
                                      - w * norm.logpdf(z, STATE_Z[f][0], STATE_SCALE[0]))))
    gbm = GradientBoostingClassifier(max_depth=3, n_estimators=150, random_state=0).fit(X, Yv)
    if _HS:
        r = _shap.TreeExplainer(gbm).shap_values(X)
        if isinstance(r, list): r = r[-1]
        r = np.asarray(r); r = r[:, :, -1] if r.ndim == 3 else r
        shap_imp = pd.Series(np.abs(r).mean(0), index=feats)
    else:
        lin = LogisticRegression(max_iter=2000).fit(X, Yv)
        shap_imp = pd.Series(np.abs(lin.coef_[0]) * X.std().values, index=feats)
    c = pd.DataFrame({"LLR": pd.Series(llr), "SHAP": shap_imp}).sort_values("SHAP", ascending=False)
    CACHE_A2.parent.mkdir(parents=True, exist_ok=True); c.to_csv(CACHE_A2)
rho = spearmanr(c["LLR"].rank(ascending=False), c["SHAP"].rank(ascending=False)).correlation
check("Fig 3a Spearman", rho)
nz = lambda s: s / (s.max() if s.max() > 0 else 1.0)
xv = np.arange(len(c)); wd = 0.4
plt.figure(figsize=(7, 4))
plt.bar(xv - wd / 2, nz(c["LLR"]), wd, label="LLR (normalized)", color=BLUE)
plt.bar(xv + wd / 2, nz(c["SHAP"]), wd, label="SHAP", color=GREEN)
plt.xticks(xv, c.index, rotation=30, ha="right"); plt.ylabel("relative importance")
plt.title(f"LLR vs SHAP (Failure vs Stable; Spearman {rho:+.2f})"); plt.legend()
plt.tight_layout(); plt.savefig(STAGE / "figure_4.png", dpi=150); plt.close()
print("Figure 3a ->", STAGE / "figure_4.png")

  check Fig 3a Spearman: +0.571 (published +0.571) -> OK
Figure 3a -> /Users/sumiyasayeed/Downloads/MSc/PEMDT - Diseases/JBI/Revision/figures_regenerated/figure_4.png


In [13]:
# ------------------------------- A3 · Figure S1 left: pre-onset AUROC by lead time (original analysis)
# Same data and style as the published plot (RF-PEMDT.ipynb, cell 26); the logistic line is dropped
# because Table S9 now reports the comparators on the revision split. Saved under the existing file name.
res = pd.read_csv(RESP_DIR / "RF_PEMDT_output" / "validation_horizon" / "fitted_hmm_horizon.csv")
plt.figure(figsize=(7, 5))
plt.plot(res["horizon_h"], res["hand_hmm"], "o-", color=BLUE, label="Clinically specified emissions (published model)")
plt.plot(res["horizon_h"], res["fitted_hmm"], "^-", color=GREEN, label="Fitted emissions (class-conditional, worst-case)")
plt.axhline(0.5, color="k", ls=":", lw=1)
plt.xlabel("Prediction horizon before ventilation (hours)")
plt.ylabel("Test AUROC"); plt.ylim(0.45, 0.75)
plt.title("Pre-onset discrimination by lead time")
plt.legend(loc="upper right"); plt.tight_layout()
plt.savefig(STAGE / "fitted_hmm_vs_all.png", dpi=130); plt.close()
print(res.round(3).to_string(index=False))
print("Figure S1 left ->", STAGE / "fitted_hmm_vs_all.png")

 horizon_h  hand_hmm  fitted_hmm  fitted_lr
         0     0.644       0.673      0.834
         2     0.630       0.656      0.865
         4     0.640       0.663      0.821
         6     0.655       0.674      0.734
        12     0.638       0.663      0.695
        24     0.627       0.650      0.689
Figure S1 left -> /Users/sumiyasayeed/Downloads/MSc/PEMDT - Diseases/JBI/Revision/figures_regenerated/fitted_hmm_vs_all.png


In [14]:
# ------------------------------------------------- B1 · Figure 3b: AKI LLR vs SHAP
# Verbatim logic of "Standalone · AKI LLR vs SHAP" (AKI_PEMDT_KDIGO.ipynb); only the title changes.
# LLR compares KDIGO 3 with no AKI (TOP = 3 vs state 0), as in the published cell.
CACHE_B1 = AKI_DIR / "figures" / "aki_llr_vs_shap_values.csv"
if CACHE_B1.exists() and not RECOMPUTE:
    c = pd.read_csv(CACHE_B1, index_col=0)
else:
    FEATURES_B = ["creatinine_ratio", "dSCr_48h", "uo_rate", "bun"]
    ANCH = {"creatinine_ratio": ([1.00, 1.60, 2.40, 3.50], [0.30, 0.35, 0.50, 0.90]),
            "dSCr_48h": ([0.00, 0.35, 0.60, 1.00], [0.15, 0.20, 0.30, 0.50]),
            "uo_rate": ([1.00, 0.45, 0.30, 0.10], [0.40, 0.18, 0.13, 0.08]),
            "bun": ([15.0, 30.0, 45.0, 70.0], [9.0, 12.0, 16.0, 22.0])}
    WTS = {"creatinine_ratio": 1.6, "dSCr_48h": 1.3, "uo_rate": 1.5, "bun": 1.0}
    TOP = 3
    df = pd.read_parquet(AKI_DIR / "derived" / "aki_cohort_v2.parquet", columns=FEATURES_B + ["kdigo_stage"])
    df = df.dropna(subset=["kdigo_stage"])
    if len(df) > 300_000:
        df = df.sample(300_000, random_state=0)
    X = df[FEATURES_B].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    for f in FEATURES_B:
        lo, hi = X[f].quantile([0.001, 0.999]); X[f] = X[f].clip(lo, hi)
    X = X.fillna(X.median())
    y = (df["kdigo_stage"].to_numpy() >= 2).astype(int)
    llr = {}
    for f in FEATURES_B:
        mu, sd = ANCH[f]; w = WTS[f]; x = X[f].to_numpy()
        llr[f] = float(np.mean(np.abs(w * (norm.logpdf(x, mu[TOP], sd[TOP]) - norm.logpdf(x, mu[0], sd[0])))))
    gbm = GradientBoostingClassifier(max_depth=3, n_estimators=150, random_state=0).fit(X, y)
    if _HS:
        r = _shap.TreeExplainer(gbm).shap_values(X)
        if isinstance(r, list): r = r[-1]
        r = np.asarray(r); r = r[:, :, -1] if r.ndim == 3 else r
        shap_imp = pd.Series(np.abs(r).mean(0), index=FEATURES_B)
    else:
        lin = LogisticRegression(max_iter=2000).fit(X, y)
        shap_imp = pd.Series(np.abs(lin.coef_[0]) * X.std().values, index=FEATURES_B)
    c = pd.DataFrame({"LLR": pd.Series(llr), "SHAP": shap_imp}).sort_values("SHAP", ascending=False)
    c.to_csv(CACHE_B1)
rho = spearmanr(c["LLR"].rank(ascending=False), c["SHAP"].rank(ascending=False)).correlation
check("Fig 3b Spearman", rho)
nz = lambda s: s / (s.max() if s.max() > 0 else 1.0)
xv = np.arange(len(c)); wd = 0.4
plt.figure(figsize=(6, 4))
plt.bar(xv - wd / 2, nz(c["LLR"]), wd, label="LLR (normalized)", color=BLUE)
plt.bar(xv + wd / 2, nz(c["SHAP"]), wd, label="SHAP", color=GREEN)
plt.xticks(xv, c.index, rotation=30, ha="right"); plt.ylabel("relative importance")
plt.title(f"LLR vs SHAP (KDIGO 3 vs no AKI; Spearman {rho:+.2f})"); plt.legend()
plt.tight_layout(); plt.savefig(STAGE / "figure_5.png", dpi=150); plt.close()
print("Figure 3b ->", STAGE / "figure_5.png")

  check Fig 3b Spearman: +0.800 (published +0.800) -> OK
Figure 3b -> /Users/sumiyasayeed/Downloads/MSc/PEMDT - Diseases/JBI/Revision/figures_regenerated/figure_5.png


In [15]:
# ------------------------------------- B2 · Figure S2 right: nephrotoxin association (adjusted)
# The Layer-3 table needs the engine's risk at hour 23 for every stay. Instead of rerunning the
# notebook, the published JAX engine is run in one fast batch (as in AKI_revision_IBMED_v2, R0c) and
# checked against the published per-stay call. The table and the estimators are the published code
# (cell "10 · LAYER 3" and cell "8f-3"); only the plot title changes.
CT_CACHE = AKI_DIR / "derived" / "cache" / "layer3_ct.parquet"
SRC = AKI_DIR / "AKI_PEMDT_KDIGO.ipynb"
if CT_CACHE.exists() and not RECOMPUTE:
    ct = pd.read_parquet(CT_CACHE)
else:
    t0 = time.time()
    os.environ["JAX_PLATFORMS"] = "cpu"              # must be set before jax is imported
    A = {"__name__": "aki_source"}                     # separate namespace for the AKI definitions
    with in_dir(AKI_DIR):
        exec(ru.cell_source(SRC, "1 · CONFIG"), A)
        exec(ru.cell_source(SRC, "2 · LAYER 1"), A)
        exec("import glob as _glob\n" + ru.defs_only(ru.cell_source(SRC, "4 · PHASE 0 ETL")), A)
        exec(ru.defs_only(ru.cell_source(SRC, "5c · COMORBIDITY FLAGS")), A)
        exec(ru.defs_only(ru.cell_source(SRC, "6 · RUN LAYER 1")), A)
        A["USE_SYNTHETIC"] = False
        jnp = A["jnp"]
        NEED = ["stay_id", "subject_id", "anchor_age", "hour", "creatinine_ratio", "dSCr_48h", "uo_rate", "bun",
                "rrt_on", "kdigo_stage", "nephrotoxin_on_board", "baseline_creatinine"]
        coh = pd.read_parquet(A["COHORT_PARQUET"], columns=NEED)
        coh = A["_add_comorbidity_flags"](coh)
        coh = A["smooth_uo_emission"](coh).reset_index(drop=True)      # sorts by stay, hour
    F = list(A["FEATURES"]); K = A["N_STATES"]
    sid = coh["stay_id"].to_numpy()
    STARTS = np.r_[0, np.nonzero(np.diff(sid))[0] + 1]; LENS = np.diff(np.r_[STARTS, len(coh)])
    AGEIDX = np.array([A["age_bracket"](a if np.isfinite(a) else 65.0)
                       for a in coh["anchor_age"].to_numpy(float)[STARTS]])
    X = np.zeros((len(coh), len(F))); M = np.zeros_like(X)
    for j, f in enumerate(F):
        raw = pd.to_numeric(coh[f], errors="coerce").to_numpy(float)
        lo, hi = A["VALID_RANGES"].get(f, (-np.inf, np.inf))
        ok = ~np.isnan(raw) & (raw > lo) & (raw < hi); X[ok, j] = raw[ok]; M[ok, j] = 1.0
    jm, js = jnp.array(A["CLINICAL_MEANS"]), jnp.array(A["CLINICAL_SCALES"])
    jw = jnp.array(np.array([A["FEATURE_WEIGHTS"][f] for f in F], float)); jp = jnp.array(A["PRIOR_BELIEF"])
    jb = [jnp.array(A["default_transition"](a)) for a in range(len(A["AGE_BRACKETS"]))]
    bucket = lambda n: int(2 ** np.ceil(np.log2(max(int(n), 8))))   # pad to powers of two (causal filter)
    B = np.empty((len(coh), K), np.float32)
    for k in range(len(STARTS)):
        st, ln = int(STARTS[k]), int(LENS[k]); Lb = bucket(ln)
        x = np.zeros((Lb, len(F))); m = np.zeros((Lb, len(F))); x[:ln] = X[st:st + ln]; m[:ln] = M[st:st + ln]
        B[st:st + ln] = np.asarray(A["forward_filter"](jnp.array(x), jnp.array(m), jm, js, jb[AGEIDX[k]], jp, jw))[:ln]
    B = np.clip(B, 0.0, 1.0)
    rrt = pd.to_numeric(coh["rrt_on"], errors="coerce").fillna(0).to_numpy() > 0
    B[rrt] = np.eye(K, dtype=np.float32)[A["ABSORBING_RRT_STATE"]]
    coh["risk_score"] = np.clip(B[:, 2] + B[:, 3], 0.0, 1.0)
    # the batch run must reproduce the published per-stay engine
    diffs = []
    for s_ in np.random.default_rng(7).choice(coh["stay_id"].unique(), 50, replace=False):
        g = coh[coh.stay_id == s_].sort_values("hour")
        r = A["run_filter_patient"](g)
        diffs.append(np.max(np.abs(r["risk_score"].to_numpy() - g["risk_score"].to_numpy())))
    print(f"engine: max |batch - published| over 50 stays = {max(diffs):.2e}")
    assert max(diffs) < 1e-6, "batch engine does not reproduce the published engine; stop and send this output"
    # Layer-3 causal table, identical to the published cell "10 · LAYER 3"
    W = A["L3"].t_window_h; rows = []
    for s_, g in coh.groupby("stay_id"):
        g = g.sort_values("hour")
        if g["hour"].max() < W + 1:
            continue
        pre = g[g["hour"] < W]; post = g[g["hour"] >= W]
        if len(pre) == 0 or len(post) == 0 or (pre["kdigo_stage"] >= 2).any():
            continue
        a = pre.iloc[-1]
        rows.append({
            "stay_id": s_,
            "T": int((pre["nephrotoxin_on_board"] > 0).any()),
            "Y": int((post["kdigo_stage"] >= 2).any()),
            "age": float(a.get("anchor_age", 65) or 65),
            "risk24": float(a.get("risk_score", 0.0) or 0.0),
            "stage24": int(a.get("kdigo_stage", 0) or 0),
            "crratio24": float(pd.to_numeric(a.get("creatinine_ratio"), errors="coerce") or 1.0),
            "bun24": float(pd.to_numeric(a.get("bun"), errors="coerce") or np.nan),
            "uo24": float(pd.to_numeric(pre["uo_rate"], errors="coerce").mean()),
            "sepsis": int(a.get("sepsis_dx", 0) or 0),
            "ckd": int(a.get("ckd", 0) or 0),
            "basecr": float(pd.to_numeric(a.get("baseline_creatinine"), errors="coerce") or np.nan),
        })
    ct = pd.DataFrame(rows)
    CT_CACHE.parent.mkdir(parents=True, exist_ok=True); ct.to_parquet(CT_CACHE, index=False)
    print(f"Layer-3 table: {len(ct):,} eligible stays; treated {ct['T'].mean():.1%}; "
          f"escalation {ct['Y'].mean():.1%} ({time.time() - t0:.0f} s)")

# estimators and plot: published cell "8f-3", unchanged except the title
from sklearn.preprocessing import StandardScaler as _SS
from sklearn.impute import SimpleImputer as _SI
_conf = [c_ for c_ in ["age", "risk24", "stage24", "crratio24", "bun24", "uo24", "sepsis", "ckd", "basecr"] if c_ in ct.columns]
d = ct.dropna(subset=["T", "Y"]).copy()
Xc = _SS().fit_transform(_SI(strategy="median").fit_transform(d[_conf]))
T = d["T"].to_numpy().astype(int); Y = d["Y"].to_numpy().astype(int)
ps = LogisticRegression(max_iter=1000).fit(Xc, T).predict_proba(Xc)[:, 1].clip(0.02, 0.98)
m1 = LogisticRegression(max_iter=1000).fit(Xc[T == 1], Y[T == 1]).predict_proba(Xc)[:, 1]
m0 = LogisticRegression(max_iter=1000).fit(Xc[T == 0], Y[T == 0]).predict_proba(Xc)[:, 1]
phi_ipw = T * Y / ps - (1 - T) * Y / (1 - ps)
phi_aipw = (m1 - m0) + T * (Y - m1) / ps - (1 - T) * (Y - m0) / (1 - ps)
def _ci(v):
    v = np.asarray(v); s = v.std() / np.sqrt(len(v)); return v.mean(), v.mean() - 1.96 * s, v.mean() + 1.96 * s
rng = np.random.default_rng(7); nb = []
for _ in range(2000):
    i = rng.integers(0, len(Y), len(Y))
    if T[i].sum() > 5 and (T[i] == 0).sum() > 5: nb.append(Y[i][T[i] == 1].mean() - Y[i][T[i] == 0].mean())
est = [("Naive", np.median(nb), np.percentile(nb, 2.5), np.percentile(nb, 97.5)),
       ("IPW", *_ci(phi_ipw)), ("AIPW (DR)", *_ci(phi_aipw))]
for name, e in zip(["S2 naive", "S2 IPW", "S2 AIPW"], est):
    check(name, e[1])
cols = [BLUE, GREEN, "#e63946"]
fig, ax = plt.subplots(figsize=(7, 3.6))
for k, (nm, e, lo, hi) in enumerate(est):
    ax.errorbar(e, 2 - k, xerr=[[e - lo], [hi - e]], fmt="o", color=cols[k], capsize=3)
    ax.text(e, 2 - k + 0.14, f"{e:+.3f} [{lo:+.3f}, {hi:+.3f}]", ha="center", fontsize=8)
ax.axvline(0, ls="--", color=GREY); ax.set_yticks([2, 1, 0]); ax.set_yticklabels([q[0] for q in est])
ax.set(xlabel="risk difference: nephrotoxin exposure -> Stage>=2", title="Nephrotoxin association (adjusted)")
plt.tight_layout(); plt.savefig(STAGE / "aki_causal_forest.png", dpi=150); plt.close()
print("Figure S2 right ->", STAGE / "aki_causal_forest.png")

Config loaded (v3.3-faithful + label/ETL fixes).
  States       : ['No_AKI', 'KDIGO_1', 'KDIGO_2', 'KDIGO_3'] | emission mode: clinical
  Output        : aki_cohort_v2.parquet (v2 — stale v1 parquet is not loaded)
  MIMIC_ROOT   : /Users/sumiyasayeed/Documents/mimic4/mimiciv (exists: True )
Layer-1 engine ported (JAX forward_filter, default_state_means/scales/transition,
run_filter_patient, compute_global_stats, attribute_window).  Devices: [CpuDevice(id=0)]
Comorbidity flags (per-stay): CKD 25.1%  |  sepsis(ICD) 29.6%
engine: max |batch - published| over 50 stays = 0.00e+00
Layer-3 table: 44,204 eligible stays; treated 37.2%; escalation 14.6% (31 s)
  check S2 naive: +0.078 (published +0.078) -> OK
  check S2 IPW: +0.030 (published +0.030) -> OK
  check S2 AIPW: +0.033 (published +0.033) -> OK
Figure S2 right -> /Users/sumiyasayeed/Downloads/MSc/PEMDT - Diseases/JBI/Revision/figures_regenerated/aki_causal_forest.png


In [16]:
# ------------------------------------------- B3 · Figure 2b and Figure S5 (top), from the R8 outputs
REV_AKI = AKI_DIR / "results" / "revision_v2"
rep = REV_AKI / "R8_recalibration_report.md"
if not rep.exists():
    print("R8 has not been run yet: run the R8 cell in AKI_revision_IBMED_v2.ipynb, then rerun this cell.")
elif "log-odds map chosen" not in rep.read_text():
    print("R8 kept the current map: Figure 2b and Figure S5 top stay as they are.")
else:
    shutil.copy(REV_AKI / "figures" / "R8_aki_calibration_24h_raw_vs_recalibrated.png", STAGE / "figure_2.png")
    GRP = pd.read_csv(REV_AKI / "R8_reliability_groups.csv")
    style = {"current (Platt on s)": ("previous map (raw score)", dict(color="0.55", marker="o", mfc="white")),
             "log-odds (Platt on logit s)": ("log-odds map (used)", dict(color=BLUE, marker="o"))}
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    for i, H in enumerate([12, 24, 48]):
        sub = GRP[(GRP["horizon_h"] == H) & GRP["map"].isin(list(style))]
        for k, (lab, st) in style.items():
            t = sub[sub["map"] == k].sort_values("g")
            ax[i].plot(t["predicted"], t["observed"], "-", lw=1.2, label=lab, **st)
        m = sub[["predicted", "observed"]].max().max() * 1.08
        ax[i].plot([0, m], [0, m], "--", color="grey", lw=1)
        ax[i].set(xlim=(0, m), ylim=(0, m), title=f"Stage ≥2 within {H} h", xlabel="Predicted", ylabel="Observed")
    ax[0].legend(loc="upper left", frameon=False)
    plt.tight_layout(); plt.savefig(STAGE / "aki_reliability_by_horizon.png", dpi=200); plt.close()
    print("R8 chose the log-odds map: Figure 2b copied and Figure S5 top redrawn in", STAGE)

R8 chose the log-odds map: Figure 2b copied and Figure S5 top redrawn in /Users/sumiyasayeed/Downloads/MSc/PEMDT - Diseases/JBI/Revision/figures_regenerated


In [17]:
# ---------------------------------------------- C1 · Figure 3c: cirrhosis LLR vs SHAP
# `cmp` comes from the published explainability cell, which needs the full-data HMM (Step 2) and the
# visit sequences (Step 1). These four cells are executed verbatim from the cirrhosis notebook in their
# own namespace; none of the revision cells (cross-validation, bootstrap, K selection) are run.
# The first run reads labevents (10 to 30 minutes); later runs use the cached table (seconds).
CACHE_C1 = CIRR_DIR / "results" / "progression_output" / "revision" / "llr_vs_shap_cmp.csv"
SRC_C = CIRR_DIR / "PEMDT_cirrhosis_progression_hmm.ipynb"
if CACHE_C1.exists() and not RECOMPUTE:
    cmp = pd.read_csv(CACHE_C1, index_col=0)
else:
    t0 = time.time(); Cn = {"__name__": "cirrhosis_source"}
    with in_dir(CIRR_DIR):
        for marker in ["Missing-aware diagonal-Gaussian HMM", "CANDIDATE_ROOTS", "STEP 1 — at-risk cohort",
                       "STEP 2 — unsupervised HMM", "EXPLAINABILITY — LLR (raw + variance-normalized) vs SHAP"]:
            print(f"--- running: {marker}")
            exec(ru.cell_source(SRC_C, marker), Cn)
    cmp = Cn["cmp"]
    CACHE_C1.parent.mkdir(parents=True, exist_ok=True); cmp.to_csv(CACHE_C1)
    print(f"cirrhosis explainability rebuilt in {time.time() - t0:.0f} s")
rho_norm = spearmanr(cmp["LLR_norm_rk"], cmp["SHAP_rk"]).correlation
check("Fig 3c Spearman", rho_norm)
# published cell "V6: LLR vs SHAP", unchanged except the title
_c = cmp.sort_values("SHAP", ascending=False)
_nz = lambda sr: sr / (sr.max() if sr.max() > 0 else 1.0)
xv = np.arange(len(_c)); w = 0.4
plt.figure(figsize=(9, 4))
plt.bar(xv - w / 2, _nz(_c["LLR_norm"]), w, label="LLR (normalized)", color=BLUE)
plt.bar(xv + w / 2, _nz(_c["SHAP"]), w, label="SHAP", color=GREEN)
plt.xticks(xv, _c.index, rotation=45, ha="right"); plt.ylabel("Relative importance")
plt.title(f"LLR vs SHAP (stage 2 vs stage 0; Spearman {rho_norm:+.2f})")
plt.legend(); plt.tight_layout()
plt.savefig(STAGE / "figure_6.png", dpi=150); plt.close()
print("Figure 3c ->", STAGE / "figure_6.png")

--- running: Missing-aware diagonal-Gaussian HMM
--- running: CANDIDATE_ROOTS
MIMIC_ROOT = /Users/sumiyasayeed/Documents/mimic4/mimiciv | hosp exists: True
Config loaded. obs = ['AST', 'ALT', 'Platelets', 'Albumin', 'Bilirubin', 'Creatinine', 'INR', 'Sodium', 'Hemoglobin', 'WBC']
--- running: STEP 1 — at-risk cohort
at-risk non-cirrhotic, not decompensated at baseline : 20,402
  with >= 2 pre-event lab visits ............ 5,597
  progress to DECOMPENSATION (events) ............ 301 (5.4%)
  median follow-up (days) ........................ 773
--- running: STEP 2 — unsupervised HMM
✅ HMM fit (5,597 patients, 2.7s); stage severity (asc): [-1.12  2.49  7.63]

Decompensation rate by baseline latent stage:
               n  event_rate  median_FU
base_stage                             
0.0         3352       0.033    919.608
1.0         1305       0.076    586.776
2.0          940       0.099    576.868
--- running: EXPLAINABILITY — LLR (raw + variance-normalized) vs SHAP
visit-level: 23,636

In [18]:
# ------------------------------------------------------ D · copy into the Revision folder (optional)
# Look at the figures in Revision/figures_regenerated first. Then set COPY_INTO_PLACE = True and rerun.
COPY_INTO_PLACE = False
TARGET = {"figure_1.png": "figure_1.png", "figure_2.png": "figure_2.png", "figure_4.png": "figure_4.png",
          "figure_5.png": "figure_5.png", "figure_6.png": "figure_6.png",
          "aki_causal_forest.png": "figures/aki_causal_forest.png",
          "fitted_hmm_vs_all.png": "figures/fitted_hmm_vs_all.png",
          "aki_reliability_by_horizon.png": "aki_reliability_by_horizon.png"}
for src, dst in TARGET.items():
    p = STAGE / src
    if not p.exists():
        print(f"  not generated: {src}"); continue
    if COPY_INTO_PLACE:
        shutil.copy(p, JBI / "Revision" / dst); print(f"  copied {src} -> Revision/{dst}")
    else:
        print(f"  ready: {src} -> Revision/{dst}")

  ready: figure_1.png -> Revision/figure_1.png
  ready: figure_2.png -> Revision/figure_2.png
  ready: figure_4.png -> Revision/figure_4.png
  ready: figure_5.png -> Revision/figure_5.png
  ready: figure_6.png -> Revision/figure_6.png
  ready: aki_causal_forest.png -> Revision/figures/aki_causal_forest.png
  ready: fitted_hmm_vs_all.png -> Revision/figures/fitted_hmm_vs_all.png
  ready: aki_reliability_by_horizon.png -> Revision/aki_reliability_by_horizon.png
